In [2]:
import sys
sys.path.append(r'C:\Users\adeeb\OneDrive\Desktop\olist-ecommerce-analysis\src')

import sqlite3
import pandas as pd
from config import DB_PATH

conn = sqlite3.connect(DB_PATH)
pd.set_option('display.max_columns', None)

In [3]:
df = pd.read_sql("""
WITH o AS (
    SELECT f.order_id,
           MAX(f.is_late)    AS is_late,
           MAX(f.delay_days) AS delay_days,
           SUM(f.price)      AS price,
           SUM(f.freight_value) AS freight,
           COUNT(*)          AS n_items
    FROM fact_order_items f
    JOIN dim_date d ON f.purchase_date_key = d.date_key
    WHERE f.order_status = 'delivered'
      AND f.is_late IS NOT NULL
      AND d.year_month BETWEEN '2017-01' AND '2018-08'
    GROUP BY f.order_id
)
SELECT o.*, r.review_score
FROM o
JOIN fact_reviews r ON o.order_id = r.order_id
""", conn)

print("Rows:", len(df), "(should be 95560)")
print(df.groupby('is_late')['review_score'].agg(['count', 'mean']).round(2))

Rows: 95560 (should be 95560)
         count  mean
is_late             
0        89182  4.29
1         6378  2.27


In [4]:
from scipy import stats
import numpy as np

late = df.loc[df['is_late'] == 1, 'review_score']
ontime = df.loc[df['is_late'] == 0, 'review_score']
diff = late.mean() - ontime.mean()

print("Mean score: late", round(late.mean(), 2), "| not late", round(ontime.mean(), 2),
      "| difference", round(diff, 2))

t, p_t = stats.ttest_ind(late, ontime, equal_var=False)
print(f"\n1. Welch t-test:    t = {t:.1f}, p = {p_t:.3e}")

u, p_u = stats.mannwhitneyu(late, ontime, alternative='two-sided')
print(f"2. Mann-Whitney U:  p = {p_u:.3e}")

table = pd.crosstab(df['is_late'], df['review_score'] <= 2)
chi2, p_c, dof, expected = stats.chi2_contingency(table)
print(f"3. Chi-square (1-2 star share): p = {p_c:.3e}")
print(table)

se = np.sqrt(late.var(ddof=1) / len(late) + ontime.var(ddof=1) / len(ontime))
print("\n95% CI for the difference:", round(diff - 1.96 * se, 2), "to", round(diff + 1.96 * se, 2))
pooled_sd = np.sqrt(((len(late) - 1) * late.var(ddof=1) + (len(ontime) - 1) * ontime.var(ddof=1))
                    / (len(late) + len(ontime) - 2))
print("Cohen's d:", round(diff / pooled_sd, 2))

Mean score: late 2.27 | not late 4.29 | difference -2.02

1. Welch t-test:    t = -100.7, p = 0.000e+00
2. Mann-Whitney U:  p = 0.000e+00
3. Chi-square (1-2 star share): p = 0.000e+00
review_score  False  True 
is_late                   
0             80934   8248
1              2398   3980

95% CI for the difference: -2.06 to -1.98
Cohen's d: -1.71
